# YOLOv11 — Shelf Void Detection (Free GPU Training)

**Roman Urdu guide (beginner ke liye):**

Yeh notebook aapke Roboflow dataset zip se **free me** YOLOv11 model train karta hai — Google Colab ke free T4 GPU pe. Zero paise, zero Roboflow credits.

## Kaise use karein (5 steps):
1. **GPU on karo** → Runtime menu → *Change runtime type* → **T4 GPU** → Save
2. **dataset.zip upload karo** → left sidebar 📁 → `/content` → Upload → `dataset.zip`
3. **Run All** karo (Ctrl+F9 ya Runtime → Run all)
4. **Wait** ~25-35 min (yolo11s, 120 epochs)
5. **best.pt download** → niche wali cells automatically download karenge

## Classes (v5-v6 dataset — auto-detected from data.yaml)
Notebook classes ko **data.yaml se automatically padhta hai**, taaki future versions (v7, v8...) ke saath bhi kaam kare. Current dataset:
- `0 = disorted` (mis-faced / distorted product)
- `1 = product` (occupied)
- `2 = shelf-void` (khali shelf / void)

> ⚠️ Class IDs positional hain — yeh order aapke Roboflow export ke `data.yaml` se aata hai, isliye notebook me **hardcode nahi kiya**.

---


## Step 1 — GPU check karo

Cell run karne pe **Tesla T4** ya koi GPU dikhni chahiye. Agar `command not found` ya CPU aaye toh: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - GPU on karo!")


## Step 2 — Ultralytics (YOLOv11) install karo

Official YOLOv11 package. Install me ~1 min lagta hai.

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import os, shutil, yaml, glob
print("Ultralytics installed")


## Step 3 — Dataset upload + setup

### Pehle zip upload karo:
Left sidebar me **folder icon 📁** → `/content` → **Upload** → apna file select karo. **Naam `dataset.zip` rakhna** (ya neeche `ZIP_NAME` badal do).

Upload ~5 min lagta hai. Upload complete hone ke baad hi yeh cell run karo.

**Download kahan se:** Roboflow → aapka project → Version → *Download Dataset* → **YOLOv8** ya **YOLOv11** format (dono same label format use karte hain — ultralytics dono padh leta hai).

In [ ]:
ZIP_NAME = "dataset.zip"   # agar file ka naam alag hai toh yahan badal do
ROOT = "/content"
DATA_DIR = ROOT + "/shelf_dataset"

# Clean previous
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

zip_path = os.path.join(ROOT, ZIP_NAME)
assert os.path.exists(zip_path), zip_path + " nahi mila! Pehle zip upload karo (sidebar 📁 se Upload)."

# Unzip
!unzip -q "{zip_path}" -d "{DATA_DIR}"
print("Unzipped to:", DATA_DIR)
print("Contents:", os.listdir(DATA_DIR))


In [ ]:
# Structure check + data.yaml path fix
# Roboflow export me paths "../train/images" ki hote hain (relative-wrong) -> hum absolute likhte hain.
# LEKIN nc + names ko original export se padhte hain (hardcode NAHI) -> labels ke saath hamesha match.
yaml_files = glob.glob(DATA_DIR + "/**/data.yaml", recursive=True)
assert yaml_files, "data.yaml nahi mila"
data_yaml_path = os.path.abspath(yaml_files[0])
base_dir = os.path.dirname(data_yaml_path)

# Roboflow classes read karo = single source of truth (future versions ke liye safe)
with open(data_yaml_path) as f:
    src = yaml.safe_load(f) or {}
names = src.get("names")
if isinstance(names, dict):                       # kuch exports {0: 'x', ...} dete hain
    names = [names[i] for i in sorted(names)]
if not names:                                     # fallback agar field missing ho
    names = ["disorted", "product", "shelf-void"]
nc = src.get("nc") or len(names)
assert nc == len(names), "nc=%s but names len=%d — data.yaml check karo" % (nc, len(names))

cfg = {
    "train": base_dir + "/train/images",
    "val":   base_dir + "/valid/images",
    "test":  base_dir + "/test/images",
    "nc": nc,
    "names": names,
}
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("data.yaml fixed:", data_yaml_path)
print(open(data_yaml_path).read())

for split in ["train", "valid", "test"]:
    imgs = glob.glob(base_dir + "/" + split + "/images/*.jpg")
    labs = glob.glob(base_dir + "/" + split + "/labels/*.txt")
    print("%-6s: %d images, %d labels" % (split, len(imgs), len(labs)))

# Sanity: koi label class id nc ke bahar toh nahi? (agar haan toh training galat/classes drop honge)
max_id = -1
for lab in glob.glob(base_dir + "/**/labels/*.txt", recursive=True):
    with open(lab) as f:
        for line in f:
            t = line.split()
            if t and t[0].lstrip("-").isdigit():
                max_id = max(max_id, int(t[0]))
print("Max class id in labels:", max_id, "| nc:", nc)
assert max_id < nc, "Label me class id %d hai but nc=%d — data.yaml me names check karo!" % (max_id, nc)
print("Class map:", {i: names[i] for i in range(nc)})


## Step 3.5 — Class Imbalance Fix (Oversampling) ⭐

Train images me `product` (~6160) aur `shelf-void` (~1240) **har image me** hain (100%). `disorted` sirf ~28% images me hai (~60 instances) — yahi **asli minority** hai. Isliye oversampling **sirf `disorted`** ko target karta hai.

> Kyun `shelf-void` nahi? Wo already **har** train image me hai, toh uska oversample karne se dataset uniform ×3 badhta hai — rebalance kuch nahi hota. Sirf `disorted` jaisi class (jo kuch images me hi hai) oversampling se genuinely boost hoti hai.

Notebook **class NAME se id resolve karta hai** (hardcoded `0`/`2` nahi), taaki Roboflow class order badal de tab bhi sahi class oversample ho. Neeche `OVERSAMPLE_CLASSES` badal sakte ho.

> ⚠️ **Valid split me `disorted` = 0 instances hai.** Isliye `disorted` ki metrics valid par nahi milengi — **test split** par dekho (Step 5). Asli fix: **zyada disorted images** collect karo.

In [ ]:
# === OVERSAMPLING: minority class wali train images ko duplicate karo ===
# Class NAME se id resolve karte hain -> Roboflow reorder hone par bhi safe.
# NOTE: shelf-void har image me hai -> uska oversample rebalance nahi karta. disorted hi real minority hai.
OVERSAMPLE_CLASSES = ["disorted"]    # in classes wali images boost karo. naam se resolve hota hai.
OVERSAMPLE_FACTOR   = 3               # image itni baar rakho (3 = original + 2 copies)

name_to_id = {n: i for i, n in enumerate(names)}
target_ids = []
for c in OVERSAMPLE_CLASSES:
    assert c in name_to_id, "'%s' class list me nahi hai. Available: %s" % (c, list(name_to_id))
    target_ids.append(name_to_id[c])
target_ids = set(target_ids)
print("Oversample class ids:", sorted(target_ids), "(%s) x%d" % (OVERSAMPLE_CLASSES, OVERSAMPLE_FACTOR))

train_img_dir = base_dir + "/train/images"
train_lab_dir = base_dir + "/train/labels"

def has_target(lab_path):
    with open(lab_path) as f:
        for line in f:
            t = line.split()
            if t and t[0].isdigit() and int(t[0]) in target_ids:
                return True
    return False

cand = []
for img in glob.glob(train_img_dir + "/*.jpg"):
    lab = os.path.join(train_lab_dir, os.path.splitext(os.path.basename(img))[0] + ".txt")
    if os.path.exists(lab) and has_target(lab):
        cand.append((img, lab))

before_n = len(glob.glob(train_img_dir + "/*.jpg"))
added = 0
for img, lab in cand:
    for k in range(OVERSAMPLE_FACTOR - 1):   # original pehle se hai, sirf copies add karo
        stem, ext = os.path.splitext(img)
        img_dup = stem + "_dup" + str(k) + ext
        lab_dup = os.path.splitext(lab)[0] + "_dup" + str(k) + ".txt"
        if not os.path.exists(img_dup):
            shutil.copy(img, img_dup)
            shutil.copy(lab, lab_dup)
            added += 1

after_n = len(glob.glob(train_img_dir + "/*.jpg"))
print("Oversample-candidate train images:", len(cand))
print("Oversample x%d: %d copies add kiye" % (OVERSAMPLE_FACTOR, added))
print("Train images: %d -> %d" % (before_n, after_n))


## Step 4 — Training (improved settings)

Minority classes (`shelf-void`, `disorted`) ki recall badhane ke liye ye settings:

| Setting | Value | Kyun |
|---|---|---|
| Model | `yolo11s.pt` (small) | Nano se zyada capacity → voids/disorted behtar seekhega |
| Epochs | `120`, patience `35` | Acha converge, jaldi early-stop na ho |
| Oversampling | ✅ Step 3.5 (`disorted` ×3) | disorted (~28% images) ko rebalance |

> Training time ~25-35 min (yolo11s). Agar **`OutOfMemory (OOM)`** aaye toh niche `BATCH = 16` ko `8` kar do.
> Fast chahiye toh `MODEL_VARIANT = "yolo11n.pt"` (par void/disorted recall thoda kam).

In [ ]:
# === CONFIG ===
MODEL_VARIANT = "yolo11s.pt"   # yolo11n.pt (fast) / yolo11s.pt (better, default) / yolo11m.pt (best, slow)
EPOCHS = 120
IMG_SIZE = 640
BATCH = 16                     # OOM aaye toh 8 kar do
PATIENCE = 35
RUN_NAME = "shelf_void_v5_3cls"

print("Training: %s | %d epochs | batch %d | %d classes | oversample x%d" % (
    MODEL_VARIANT, EPOCHS, BATCH, nc, OVERSAMPLE_FACTOR))

model = YOLO(MODEL_VARIANT)
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=RUN_NAME,
    patience=PATIENCE,
    cache=True,
    plots=True,
    exist_ok=True,
)
print("✅ TRAINING DONE")


## Step 5 — Results + Metrics dekho

Sabse important: **`shelf-void`** class ka recall (model khali shelf kitni baar pakadta hai) aur **`disorted`** ka mAP (rarest class). Per-class numbers trained model ke `model.names` se aate hain — hardcode nahi.

> Note: valid split me `disorted` nahi hai, isliye ye eval **test split** par hota hai (jisme disorted hai).

In [ ]:
# Validation on test set
metrics = model.val(data=data_yaml_path, split="test", plots=True)
print("\n=== OVERALL ===")
print("mAP50      : %.4f" % metrics.box.map50)
print("mAP50-95   : %.4f" % metrics.box.map)
print("Precision  : %.4f" % metrics.box.mp)
print("Recall     : %.4f" % metrics.box.mr)

print("\n=== PER-CLASS (trained model.names se) ===")
cls_names = [model.names[i] for i in range(len(model.names))]
for i, n in enumerate(cls_names):
    print("%-12s: mAP50=%.4f  P=%.4f  R=%.4f" % (n, metrics.box.ap50(i), metrics.box.p[i], metrics.box.r[i]))


## Step 6 — Best Model Download

In [ ]:
best_model_path = "runs/detect/" + RUN_NAME + "/weights/best.pt"
assert os.path.exists(best_model_path), "best.pt nahi mila — training check karo"

from google.colab import files
files.download(best_model_path)
print("best.pt download ho raha hai... (browser popup)")
print("Size: %.1f MB" % (os.path.getsize(best_model_path)/1024/1024))


## Step 7 — Training Graphs

In [ ]:
from IPython.display import Image, display
run_dir = "runs/detect/" + RUN_NAME
for img_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = run_dir + "/" + img_name
    if os.path.exists(p):
        print("\n--- " + img_name + " ---")
        display(Image(filename=p, width=600))


---
*Notebook • YOLOv11 • Free Colab GPU • 3-class (disorted / product / shelf-void) • classes auto-read from data.yaml • class-imbalance oversampling*